In [1]:
import pandas as pd
import sqlite3
import os

# Tạo database từ 5 CSV files
os.chdir(os.path.expanduser("~/portfolio"))
conn = sqlite3.connect("data/portfolio.db")

# Load tất cả CSV vào database
tables = {
    "rba_decisions":    "data/raw/rba_decisions.csv",
    "sentiment_index":  "data/raw/sentiment_index.csv",
    "google_trends":    "data/raw/google_trends.csv",
    "abs_labour_force": "data/raw/abs_labour_force.csv",
    "jsa_projections":  "data/raw/jsa_projections.csv",
}

for table, path in tables.items():
    df = pd.read_csv(path)
    df.to_sql(table, conn, if_exists="replace", index=False)
    print(f"Loaded {table}: {len(df)} rows")

print("\nDatabase ready: data/portfolio.db")

Loaded rba_decisions: 139 rows
Loaded sentiment_index: 138 rows
Loaded google_trends: 552 rows
Loaded abs_labour_force: 900 rows
Loaded jsa_projections: 19 rows

Database ready: data/portfolio.db


In [3]:
# Project 1: RBA decisions overview
query = """
SELECT 
    decision,
    COUNT(*) as count,
    MIN(new_rate) as min_rate,
    MAX(new_rate) as max_rate,
    ROUND(AVG(ABS(rate_change)), 2) as avg_change
FROM rba_decisions
WHERE decision != 'hold'
GROUP BY decision
"""
print("=== RBA Decisions Summary ===")
print(pd.read_sql(query, conn))

# Sentiment overview
query2 = """
SELECT 
    ROUND(AVG(sentiment_score), 1) as avg_sentiment,
    MIN(sentiment_score) as lowest,
    MAX(sentiment_score) as highest,
    SUM(CASE WHEN is_pessimistic = 1 THEN 1 ELSE 0 END) as pessimistic_months,
    MAX(pessimism_streak) as longest_streak
FROM sentiment_index
"""
print("\n=== ANZ Consumer Confidence Summary ===")
print(pd.read_sql(query2, conn))

=== RBA Decisions Summary ===
  decision  count  min_rate  max_rate  avg_change
0      cut     13      0.25      4.35        0.24
1     hike     16      0.10      4.10        0.31

=== ANZ Consumer Confidence Summary ===
   avg_sentiment  lowest  highest  pessimistic_months  longest_streak
0          100.1    63.6    121.4                  60              52


In [5]:
query3 = """
SELECT 
    r.date as hike_date,
    r.new_rate,
    r.rate_change,
    s_before.sentiment_score as sentiment_before,
    s_after1.sentiment_score as sentiment_1m_after,
    s_after3.sentiment_score as sentiment_3m_after,
    ROUND(s_after1.sentiment_score - s_before.sentiment_score, 1) as drop_1m,
    ROUND(s_after3.sentiment_score - s_before.sentiment_score, 1) as drop_3m
FROM rba_decisions r
LEFT JOIN sentiment_index s_before 
    ON strftime('%Y-%m', s_before.date) = strftime('%Y-%m', r.date)
LEFT JOIN sentiment_index s_after1 
    ON s_after1.date = date(r.date, '+1 month')
LEFT JOIN sentiment_index s_after3 
    ON s_after3.date = date(r.date, '+3 months')
WHERE r.decision = 'hike'
ORDER BY r.date
"""
print("=== Sentiment Change After Each Rate Hike ===")
df_event = pd.read_sql(query3, conn)
print(df_event.to_string(index=False))
print(f"\nAverage drop 1 month after hike: {df_event['drop_1m'].mean():.1f} pts")
print(f"Average drop 3 months after hike: {df_event['drop_3m'].mean():.1f} pts")

=== Sentiment Change After Each Rate Hike ===
 hike_date  new_rate  rate_change  sentiment_before  sentiment_1m_after  sentiment_3m_after  drop_1m  drop_3m
2022-05-02      0.10         0.25              90.3                 NaN                 NaN      NaN      NaN
2022-06-01      0.35         0.50              85.0                82.8                85.7     -2.2      0.7
2022-07-01      0.85         0.50              82.8                83.6                84.2      0.8      1.4
2022-08-01      1.35         0.50              83.6                85.7                80.2      2.1     -3.4
2022-09-01      1.85         0.50              85.7                84.2                82.8     -1.5     -2.9
2022-10-03      2.35         0.25              84.2                 NaN                 NaN      NaN      NaN
2022-11-01      2.60         0.25              80.2                82.8                80.5      2.6      0.3
2022-12-01      2.85         0.25              82.8                86.9   

In [7]:
query4 = """
SELECT 
    j.industry,
    j.baseline_2025,
    j.projected_new_jobs_000s,
    j.annual_growth_pct,
    a.employed_000s as current_employed,
    a.yoy_growth_pct as actual_yoy_growth
FROM jsa_projections j
LEFT JOIN (
    SELECT industry, employed_000s, yoy_growth_pct
    FROM abs_labour_force
    WHERE date = (SELECT MAX(date) FROM abs_labour_force)
) a ON a.industry LIKE '%' || j.industry || '%'
   OR j.industry LIKE '%' || a.industry || '%'
ORDER BY j.annual_growth_pct DESC
"""
print("=== Industry Growth: Projected vs Actual ===")
df_workforce = pd.read_sql(query4, conn)
print(df_workforce.to_string(index=False))

=== Industry Growth: Projected vs Actual ===
                                       industry  baseline_2025  projected_new_jobs_000s  annual_growth_pct  current_employed  actual_yoy_growth
               Healthcare and Social Assistance    2369.311531                   290.33               2.34               NaN                NaN
Professional, Scientific and Technical Services    1351.631559                   136.64               1.94            1424.9              12.36
               Public Administration and Safety     988.981828                    73.15               1.44             967.5              -2.49
        Rental, Hiring and Real Estate Services     244.535538                    17.48               1.39             208.9             -15.97
                                   Construction    1347.967556                    93.53               1.35            1370.6              -0.38
               Financial and Insurance Services     572.077626                    36.50    

In [9]:
# Fix: join trực tiếp bằng tên chính xác
abs_latest = pd.read_sql("""
    SELECT industry, employed_000s, yoy_growth_pct
    FROM abs_labour_force
    WHERE date = (SELECT MAX(date) FROM abs_labour_force)
    AND industry != 'Employed total ;'
    ORDER BY yoy_growth_pct DESC
""", conn)

print("=== ABS Latest Employment (Actual) ===")
print(abs_latest.to_string(index=False))

=== ABS Latest Employment (Actual) ===
                                       industry  employed_000s  yoy_growth_pct
     Electricity, Gas, Water and Waste Services          214.3           13.33
Professional, Scientific and Technical Services         1424.9           12.36
                                 Other Services          581.1            6.84
              Transport, Postal and Warehousing          782.6            6.66
                         Education and Training         1330.8            5.86
              Health Care and Social Assistance         2394.4            3.61
            Administrative and Support Services          439.2            3.05
                                   Retail Trade         1335.8            0.68
              Agriculture, Forestry and Fishing          294.9            0.31
                Accommodation and Food Services          962.2            0.03
                                   Construction         1370.6           -0.38
             

In [11]:
# Tìm period nào gây ra 52-month streak
query6 = """
SELECT 
    date,
    sentiment_score,
    pessimism_streak,
    mom_change
FROM sentiment_index
WHERE pessimism_streak >= 1
ORDER BY date
"""
df_streak = pd.read_sql(query6, conn)

# Tìm start và end của streak dài nhất
max_streak = df_streak['pessimism_streak'].max()
peak_row = df_streak[df_streak['pessimism_streak'] == max_streak].iloc[0]
start_date = pd.to_datetime(peak_row['date']) - pd.DateOffset(months=int(max_streak)-1)

print(f"Longest pessimism streak: {max_streak} months")
print(f"Started: {start_date.strftime('%b %Y')}")
print(f"Peaked at: {peak_row['date']}")
print(f"\nLowest confidence during streak:")
worst = df_streak[df_streak['pessimism_streak'] > 0].nsmallest(5, 'sentiment_score')[['date','sentiment_score','pessimism_streak']]
print(worst.to_string(index=False))

Longest pessimism streak: 52 months
Started: Mar 2022
Peaked at: 2026-06-01

Lowest confidence during streak:
      date  sentiment_score  pessimism_streak
2026-04-01             63.6                50
2026-05-01             66.0                51
2026-03-01             70.5                49
2026-06-01             71.4                52
2023-07-01             73.8                17


In [13]:
# Lag analysis: Google Trends spike sau RBA hike bao lâu?
query7 = """
SELECT 
    g.keyword,
    g.date,
    g.search_volume,
    g.is_spike,
    r.decision,
    r.new_rate
FROM google_trends g
LEFT JOIN rba_decisions r 
    ON strftime('%Y-%m', g.date) = strftime('%Y-%m', r.date)
WHERE g.keyword IN ('mortgage stress', 'cost of living', 'RBA interest rate')
ORDER BY g.keyword, g.date
"""
df_trends = pd.read_sql(query7, conn)

# Tìm peak search volume sau mỗi hike
hikes = pd.read_sql("SELECT date FROM rba_decisions WHERE decision='hike'", conn)
hikes['date'] = pd.to_datetime(hikes['date'])

print("=== Search Volume Spikes by Keyword ===")
for keyword in ['mortgage stress', 'cost of living', 'RBA interest rate']:
    kw_data = df_trends[df_trends['keyword'] == keyword].copy()
    kw_data['date'] = pd.to_datetime(kw_data['date'])
    spikes = kw_data[kw_data['is_spike'] == 1]
    avg_vol = kw_data['search_volume'].mean()
    max_vol = kw_data['search_volume'].max()
    max_date = kw_data.loc[kw_data['search_volume'].idxmax(), 'date']
    print(f"\n{keyword}:")
    print(f"  Average volume : {avg_vol:.1f}")
    print(f"  Peak volume    : {max_vol:.1f} ({max_date.strftime('%b %Y')})")
    print(f"  Spike months   : {len(spikes)}")

=== Search Volume Spikes by Keyword ===

mortgage stress:
  Average volume : 14.8
  Peak volume    : 48.0 (Oct 2017)
  Spike months   : 17


ValueError: attempt to get argmax of an empty sequence

In [15]:
# Kiểm tra keywords thực tế trong database
print(pd.read_sql("SELECT DISTINCT keyword FROM google_trends", conn))

               keyword
0    RBA interest rate
1  fixed rate mortgage
2      mortgage stress
3  refinance home loan


In [17]:
# Fix: dùng đúng keywords có trong database
for keyword in ['mortgage stress', 'fixed rate mortgage', 'RBA interest rate', 'refinance home loan']:
    kw_data = df_trends[df_trends['keyword'] == keyword].copy()
    
    if len(kw_data) == 0:
        print(f"\n{keyword}: no data")
        continue
        
    kw_data['date'] = pd.to_datetime(kw_data['date'])
    kw_data = kw_data.dropna(subset=['search_volume'])
    
    if len(kw_data) == 0:
        continue
    
    spikes = kw_data[kw_data['is_spike'] == 1]
    avg_vol  = kw_data['search_volume'].mean()
    max_idx  = kw_data['search_volume'].idxmax()
    max_vol  = kw_data.loc[max_idx, 'search_volume']
    max_date = kw_data.loc[max_idx, 'date']
    
    print(f"\n{keyword}:")
    print(f"  Average volume : {avg_vol:.1f}")


mortgage stress:
  Average volume : 14.8

fixed rate mortgage: no data

RBA interest rate:
  Average volume : 15.3

refinance home loan: no data


In [19]:
# Kiểm tra tất cả keywords và sample data
df_all = pd.read_sql("""
    SELECT keyword, COUNT(*) as rows, 
           ROUND(AVG(search_volume),1) as avg_vol,
           MAX(search_volume) as max_vol,
           SUM(CASE WHEN is_spike=1 THEN 1 ELSE 0 END) as spikes
    FROM google_trends
    GROUP BY keyword
""", conn)
print(df_all.to_string(index=False))

            keyword  rows  avg_vol  max_vol  spikes
  RBA interest rate   138     15.4     49.3      17
fixed rate mortgage   138     31.7     88.0       4
    mortgage stress   138     14.9     48.0      17
refinance home loan   138     42.2     97.0       0


In [21]:
# Peak dates cho từng keyword
query_peak = """
SELECT 
    keyword,
    date,
    search_volume,
    is_spike
FROM google_trends g1
WHERE search_volume = (
    SELECT MAX(search_volume) 
    FROM google_trends g2 
    WHERE g2.keyword = g1.keyword
)
ORDER BY search_volume DESC
"""
print("=== Peak Search Month Per Keyword ===")
print(pd.read_sql(query_peak, conn).to_string(index=False))

# Correlation: RBA hike months vs search spikes
query_corr = """
SELECT 
    strftime('%Y-%m', g.date) as month,
    g.keyword,
    g.search_volume,
    g.is_spike,
    r.decision,
    r.rate_change
FROM google_trends g
LEFT JOIN rba_decisions r 
    ON strftime('%Y-%m', g.date) = strftime('%Y-%m', r.date)
WHERE g.keyword = 'RBA interest rate'
AND (r.decision = 'hike' OR g.is_spike = 1)
ORDER BY g.date
"""
print("\n=== RBA Hike Months vs Search Spikes ===")
print(pd.read_sql(query_corr, conn).to_string(index=False))

=== Peak Search Month Per Keyword ===
            keyword       date  search_volume  is_spike
refinance home loan 2026-05-01           97.0         0
fixed rate mortgage 2026-05-01           88.0         0
  RBA interest rate 2023-07-01           49.3         0
    mortgage stress 2017-10-01           48.0         0

=== RBA Hike Months vs Search Spikes ===
  month           keyword  search_volume  is_spike decision  rate_change
2015-05 RBA interest rate           17.3         1      cut        -0.25
2015-11 RBA interest rate           11.3         1     hold         0.00
2016-05 RBA interest rate           11.7         1      cut        -0.25
2019-05 RBA interest rate           10.3         1     hold         0.00
2019-06 RBA interest rate           13.7         1      cut        -0.25
2019-07 RBA interest rate           19.3         1      cut        -0.25
2020-03 RBA interest rate           14.3         1      cut        -0.25
2020-03 RBA interest rate           14.3         1      